In [7]:
import nltk
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize


nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

# Corpus
documents = [
    "Apple grows on trees.",
    "Orange is a citrus fruit.",
    "Apple is also a tech company.",
    "Trees are vital for environment.",
    "Fruit salads are delicious with oranges.",
    "Orange grows on citrus trees.",
    "Apple's latest product has been a major success."
]

query = "Apple company trees"

# Step 1: Preprocessing
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess(text):
    # a. Tokenize, b. Remove stop words, c. Perform stemming
    # We also remove punctuation by checking string.isalnum()
    tokens = word_tokenize(text.lower())
    processed_tokens = [stemmer.stem(w) for w in tokens if w.isalnum() and w not in stop_words]
    return " ".join(processed_tokens)

processed_docs = [preprocess(doc) for doc in documents]
processed_query = preprocess(query)

#  Step 2: Term Document Matrix
# d. Create the vocabulary V and a. Use raw frequencies
vectorizer = CountVectorizer()
term_doc_matrix = vectorizer.fit_transform(processed_docs)
vocabulary = vectorizer.get_feature_names_out()

df_tdm = pd.DataFrame(term_doc_matrix.toarray(), columns=vocabulary,
                      index=[f"D{i+1}" for i in range(len(documents))])

print(" 1. Vocabulary V ")
print(vocabulary)
print(" 2. Term-Document Matrix  ")
print(df_tdm)

# Step 3: TF-IDF Weighting
# a. Compute TF-IDF, b. Update the matrix
tfidf_transformer = TfidfTransformer()
tfidf_matrix = tfidf_transformer.fit_transform(term_doc_matrix)

df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=vocabulary,
                        index=[f"D{i+1}" for i in range(len(documents))])

print("3. TF-IDF Term-Document Matrix ")
print(df_tfidf.round(4))

# Step 4: Cosine Similarity
# a. Represent Query Q as a vector using the same TF-IDF space
query_tdm = vectorizer.transform([processed_query])
query_tfidf = tfidf_transformer.transform(query_tdm)

# b. Compute cosine similarity between Q and all documents
cosine_sims = cosine_similarity(query_tfidf, tfidf_matrix).flatten()

print(" 4. Query Vector (TF-IDF) ")
df_query = pd.DataFrame(query_tfidf.toarray(), columns=vocabulary, index=["Q"])
print(df_query.round(4))

# Step 5: Ranking
# a. Rank the documents
results = pd.DataFrame({
    'Document': [f"D{i+1}" for i in range(len(documents))],
    'Original Text': documents,
    'Cosine Similarity': cosine_sims
})

results_ranked = results.sort_values(by='Cosine Similarity', ascending=False).reset_index(drop=True)

print(" 5. Document Ranking ")
print(results_ranked[['Document', 'Cosine Similarity']])

 1. Vocabulary V 
['also' 'appl' 'citru' 'compani' 'delici' 'environ' 'fruit' 'grow'
 'latest' 'major' 'orang' 'product' 'salad' 'success' 'tech' 'tree'
 'vital']
 2. Term-Document Matrix  
    also  appl  citru  compani  delici  environ  fruit  grow  latest  major  \
D1     0     1      0        0       0        0      0     1       0      0   
D2     0     0      1        0       0        0      1     0       0      0   
D3     1     1      0        1       0        0      0     0       0      0   
D4     0     0      0        0       0        1      0     0       0      0   
D5     0     0      0        0       1        0      1     0       0      0   
D6     0     0      1        0       0        0      0     1       0      0   
D7     0     1      0        0       0        0      0     0       1      1   

    orang  product  salad  success  tech  tree  vital  
D1      0        0      0        0     0     1      0  
D2      1        0      0        0     0     0      0  
D3      0

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


# 1. Preprocessing

Standardizing our text corpus requires tokenization followed by stop word removal and Porter stemming. This pipeline reduces all terms to their morphological base roots and filters out low-signal syntactic words. The resulting clean tokens establish the global vocabulary for our vector space model.

# 2. Term Document Matrix
Using our global vocabulary we map the dataset into a sparse matrix architecture. The rows represent our distinct documents and the columns represent our vocabulary features. The initial integer values capture the absolute frequency of each token per text.

# 3. TF-IDF Weighting
Raw frequencies disproportionately favor common terms so we apply term frequency inverse document frequency weighting. This statistical transformation penalizes features that appear across many texts while amplifying the signal of rare informative words. The transformed matrix provides a robust mathematical representation of the corpus.

# 4. Cosine Similarity
To evaluate relevance we project our search query into the exact same feature space and compute the dot product of the normalized vectors. This geometric operation measures the angular distance between the vectors rather than their absolute magnitude. A smaller angle yields a similarity score closer to one indicating strong semantic alignment.

# 5. Ranking
The algorithm identifies the third document and the first document as the most relevant matches for our query. The third document secures the top rank because it contains the highly weighted rare term compani. This high inverse document frequency gives its vector greater mathematical precedence over documents that only match common terms like tree.